<a href="https://colab.research.google.com/github/roblei007/Aula-Projeto1/blob/aprendendo-numpy/exercicios_aula_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2 - **NumPy:** refatorando o Sistema de Triagem

Curso: Técnicas de Programação (Python) — Trilha Analista de Dados

Na Aula 1 e Aula 2 construímos o Sistema de Triagem de Candidatos inteiramente em Python puro. Ele funciona, mas tem dois problemas que aparecem quando o banco cresce:

1. **Desempenho.** As análises usam `for` em Python puro. Para 10 candidatos, tudo bem. Para 10.000, começa a pesar.
2. **Capacidade analítica.** Calcular desvio padrão, percentis ou correlação em Python puro dá muito código.

Nesta aula reescrevemos as partes analíticas do sistema usando **NumPy** — sem mudar o que já funciona bem. O sistema continua o mesmo. Mudamos apenas as engrenagens internas das funções de análise.

Trabalharemos em uma branch separada (`v2-numpy`) para preservar a v1 intacta. Ao final, fazemos o merge na `main`.

## Ponto de partida

A célula abaixo prepara o ambiente:

- Importa NumPy e outras bibliotecas necessárias
- Garante que as funções da v1 estão disponíveis (caso não tenha seu arquivo da aula anterior)
- Cria dois bancos: `banco_exemplo` (5 candidatos, para testes) e `banco_grande` (10.000 candidatos, para medir desempenho)
- Guarda a versão antiga de `media_pretensao` como `media_pretensao_v1`, para comparar com a nova

In [ ]:
import numpy as np
import time
import random

# Funções da v1 — caso você não tenha o arquivo da aula anterior carregado
def classificar_nivel(anos):
    if anos < 0:  return "Inválido"
    if anos <= 2: return "Júnior"
    if anos <= 5: return "Pleno"
    return "Sênior"

def criar_candidato(nome, cargo, anos, pretensao, modalidade):
    return {
        "nome": nome, "cargo_desejado": cargo,
        "anos_experiencia": anos, "pretensao_salarial": pretensao,
        "modalidade": modalidade
    }

def adicionar_candidato(banco, candidato):
    return banco + [candidato]

# Versão antiga, mantida para comparação de desempenho
def media_pretensao_v1(banco):
    return round(sum(c["pretensao_salarial"] for c in banco) / len(banco), 2)


# Banco pequeno para testes
banco_exemplo = []
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Ana Lima",     "Analista de Dados",   1, 55000,  "remoto"))
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Bruno Costa",  "Cientista de Dados",  4, 95000,  "híbrido"))
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Carla Mendes", "Engenheira de Dados", 8, 145000, "presencial"))
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Diego Souza",  "Analista de Dados",   2, 60000,  "remoto"))
banco_exemplo = adicionar_candidato(banco_exemplo, criar_candidato("Elena Ferraz", "Cientista de Dados",  5, 100000, "híbrido"))

# Banco grande para medir desempenho
random.seed(42)
cargos      = ["Analista de Dados", "Cientista de Dados", "Engenheira de Dados", "Analista de BI"]
modalidades = ["remoto", "presencial", "híbrido"]
banco_grande = [
    criar_candidato(
        f"Pessoa {i}",
        random.choice(cargos),
        random.randint(0, 15),
        random.randint(40000, 200000),
        random.choice(modalidades)
    )
    for i in range(10_000)
]

print(f"banco_exemplo: {len(banco_exemplo)} candidatos")
print(f"banco_grande:  {len(banco_grande)} candidatos")

banco_exemplo: 5 candidatos
banco_grande:  10000 candidatos


## 1. Refatorando `media_pretensao` com NumPy

**Problema na v1:** o cálculo da média usa `sum(...) / len(...)` em Python puro. Para 10.000 registros ou mais, isso fica lento.

**Conceito de NumPy:** se convertemos uma lista para um array NumPy, ganhamos acesso a operações otimizadas:

```
np.array([55000, 95000, 145000]).mean()  ->  98333.33
```

Reescreva `media_pretensao` extraindo os salários para um `np.array` e usando o método `.mean()`. A assinatura é a mesma; muda apenas a implementação.

**Exemplo:**

```
media_pretensao(banco_exemplo)  ->  91000.0
```

In [ ]:
banco_grande[9]

{'nome': 'Pessoa 9',
 'cargo_desejado': 'Engenheira de Dados',
 'anos_experiencia': 11,
 'pretensao_salarial': 198263,
 'modalidade': 'presencial'}

In [ ]:
for c in banco_grande[:3]:
  print(c["pretensao_salarial"])

112097
66868
48331


In [ ]:
# Versão antiga, mantida para comparação de desempenho
def media_pretensao_v1(banco):
    return round(sum(c["pretensao_salarial"] for c in banco) / len(banco), 2)

In [ ]:
def media_pretensao(banco):
    banco_salarios = [c["pretensao_salarial"] for c in banco]
    np_array = np.array(banco_salarios)
    return round(np_array.mean(), 2)


In [ ]:
# Teste de correção
print(media_pretensao(banco_exemplo) == 91000.0)

# Teste de desempenho: 100 chamadas no banco grande
n = 100

inicio = time.time()
for _ in range(n):
    media_pretensao_v1(banco_grande)
tempo_v1 = (time.time() - inicio) / n * 1000

inicio = time.time()
for _ in range(n):
    media_pretensao(banco_grande)
tempo_v2 = (time.time() - inicio) / n * 1000

print(f"v1 (Python puro): {tempo_v1:.2f} ms por chamada")
print(f"v2 (NumPy):       {tempo_v2:.2f} ms por chamada")

True
v1 (Python puro): 0.37 ms por chamada
v2 (NumPy):       0.67 ms por chamada


**`Sugestão de commit:`**


refactor: usa NumPy em media_pretensao

Substitui o cálculo manual da média (sum/len) pela operação vetorizada do NumPy. Comportamento idêntico, mas pavimenta o caminho para análises mais ricas nas próximas funções.


## 2. Extratores de colunas numéricas



**Problema de organização:** toda análise vai precisar de algum dado numérico do banco (salários, anos de experiência). Repetir a extração em cada função é ruim.

**Solução:** criar funções auxiliares que extraem cada coluna como um `np.array`.

Implemente:

- `extrair_salarios(banco)` retorna um `np.array` com a pretensão salarial de cada candidato
- `extrair_experiencia(banco)` retorna um `np.array` com os anos de experiência

**Exemplo:**

```
extrair_salarios(banco_exemplo)
->  array([ 55000,  95000, 145000,  60000, 100000])

extrair_experiencia(banco_exemplo)
->  array([1, 4, 8, 2, 5])
```

In [ ]:
def extrair_salarios(banco):
  return np.array([c['pretensao_salarial'] for c in banco])

def extrair_experiencia(banco):
    return np.array([c['anos_experiencia'] for c in banco])


In [ ]:
# Testes
salarios   = extrair_salarios(banco_exemplo)
experiencia = extrair_experiencia(banco_exemplo)

print(isinstance(salarios, np.ndarray))
print(isinstance(experiencia, np.ndarray))
print(list(salarios)    == [55000, 95000, 145000, 60000, 100000])
print(list(experiencia) == [1, 4, 8, 2, 5])

True
True
True
True


**Sugestão de commit:**


feat: adiciona funcoes extratoras de colunas numericas
Centraliza a extração de salários e experiência em arrays NumPy. Próximas análises passam a operar sobre arrays prontos, sem duplicar a conversão em cada função.


## 3. Estatísticas completas




**Problema na v1:** o sistema só calculava a média. Para um analista de dados, isso é pouco — precisamos ver dispersão, valores extremos e percentis para entender a distribuição.

**Conceito de NumPy:** o array tem métodos prontos para a maioria das estatísticas:

```
arr.mean()             # média
arr.std()              # desvio padrão
arr.min()              # mínimo
arr.max()              # máximo
np.median(arr)         # mediana
np.percentile(arr, 25) # percentil 25 (1º quartil)
np.percentile(arr, 75) # percentil 75 (3º quartil)
```

Implemente `estatisticas_completas(banco)` que retorna um dicionário com todas as estatísticas acima sobre a pretensão salarial, cada valor arredondado para 2 casas decimais.

**Exemplo:**

```
estatisticas_completas(banco_exemplo)
->  {"media": 91000.0, "mediana": 95000.0, "std": 32465.37,
     "minimo": 55000.0, "maximo": 145000.0,
     "p25": 60000.0,    "p75": 100000.0}
```

In [ ]:
def estatisticas_completas(banco):
  arr = extrair_salarios(banco)

  return {
    "contagem": len(arr),
    "media": arr.mean(),
    "std": arr.std(),
    "minimo": arr.min(),
    "maximo": arr.max(),
    "mediana": np.median(arr),
    "p25": np.percentile(arr, 25),
    "p75": np.percentile(arr, 75)
  }


In [ ]:
# Testes
r = estatisticas_completas(banco_exemplo)

print(r["media"]   == 91000.0)
print(r["mediana"] == 95000.0)
print(r["minimo"]  == 55000.0)
print(r["maximo"]  == 145000.0)
print(r["p25"]     == 60000.0)
print(r["p75"]     == 100000.0)
print(abs(r["std"] - 32465.37) < 0.5)

True
True
True
True
True
True
True


**Sugestão de commit:**


feat: adiciona estatisticas_completas com desvio e percentis

A v1 só calculava a média. Esta função entrega o quadro completo (media, mediana, desvio, mínimo, máximo, p25, p75), usando os métodos otimizados do NumPy. Mesma performance, muito mais
informação para o analista.



## 4. Reajuste salarial vetorizado

**Problema na v1:** o RH pede um reajuste de 10% em todas as pretensões salariais. Em Python puro, isso exige um `for` que cria um novo dicionário para cada candidato.

**Conceito de NumPy:** operações entre um array e um escalar são aplicadas elemento a elemento — sem `for` explícito:

```
np.array([100, 200, 300]) * 1.10   ->  array([110.0, 220.0, 330.0])
```

Implemente `aplicar_reajuste(banco, percentual)` que retorna um **novo** banco com a `pretensao_salarial` de cada candidato reajustada pelo percentual informado. O banco original não deve ser modificado.

**Exemplo:**

```
novo_banco = aplicar_reajuste(banco_exemplo, 10)
novo_banco[0]["pretensao_salarial"]  ->  60500.0   # 55000 * 1.10
banco_exemplo[0]["pretensao_salarial"] ->  55000   # original intacto
```

**Orientação:** use `extrair_salarios` para pegar o array, aplique o reajuste vetorizado e depois reconstrua o banco. Use list comprehension com `zip` para juntar os candidatos aos novos salários — `{**c, "pretensao_salarial": novo_valor}` cria um dicionário novo copiando o anterior e sobrescrevendo um campo.

In [ ]:
def aplicar_reajuste(banco, percentual):
    arr = extrair_salarios(banco)
    return [{**c, "pretensao_salarial": novo_valor} for c, novo_valor in zip(banco, arr * (1 + percentual / 100))]

In [ ]:
# Testes
novo = aplicar_reajuste(banco_exemplo, 10)

# O novo banco tem os salários reajustados
print(novo[0]["pretensao_salarial"] == 60500.0)
print(novo[2]["pretensao_salarial"] == 159500.0)

# O banco original permanece intacto
print(banco_exemplo[0]["pretensao_salarial"] == 55000)
print(banco_exemplo[2]["pretensao_salarial"] == 145000)

# Reajuste negativo (corte de 5%) também funciona
corte = aplicar_reajuste(banco_exemplo, -5)
print(corte[0]["pretensao_salarial"] == 52250.0)

False
True
True
True
True


**Sugestão de commit:**


feat: adiciona aplicar_reajuste com operacao vetorizada

Permite aplicar reajustes percentuais a todo o banco em uma
única operação NumPy, sem mutar o banco original. A versão
funcional (retorna novo banco) facilita simular cenários sem
risco de corromper os dados originais.


## 5. Filtros por máscara booleana

**Problema na v1:** filtrar candidatos por um valor calculado (como "acima da média") exige calcular a média primeiro, depois iterar de novo comparando cada um.

**Conceito de NumPy:** uma comparação aplicada a um array gera uma **máscara booleana** — um array de `True`/`False`:

```
salarios = np.array([55000, 95000, 145000, 60000, 100000])
salarios > salarios.mean()
->  array([False, True, True, False, True])
```

A máscara pode ser usada para selecionar elementos do próprio array, ou — combinada com `zip` — para selecionar registros completos do banco.

Implemente `candidatos_acima_da_media(banco)` que retorna a lista de candidatos cuja pretensão salarial está acima da média do banco.

**Exemplo:**

```
candidatos_acima_da_media(banco_exemplo)
->  [<dict do Bruno>, <dict da Carla>, <dict da Elena>]
```

**Orientação:** combine `extrair_salarios`, comparação com `.mean()`, e um `zip` com list comprehension para devolver os dicionários originais.

In [ ]:
def candidatos_acima_da_media(banco):
    arr = extrair_salarios(banco)
    media = arr.mean()
    mascara = arr > media
    # return arr, media, mascara
    return [c for c, esta_acima in zip(banco, mascara) if esta_acima]


In [ ]:
# Testes
acima = candidatos_acima_da_media(banco_exemplo)
nomes_acima = sorted([c["nome"] for c in acima])

print(nomes_acima == ["Bruno Costa", "Carla Mendes", "Elena Ferraz"])
print(len(acima) == 3)

True
True


**Sugestão de commit:**

```
feat: adiciona filtro por candidatos acima da media

Usa mascara booleana do NumPy para identificar quem esta acima
da media, e zip para devolver os registros completos. Resolve
o caso, mas evidencia o limite do NumPy: filtrar registros
mistos (dicts) ainda exige um zip manual. A proxima versao
(v3-pandas) elimina essa friccao.
```

## 6. Correlação entre experiência e salário

**Capacidade nova:** existe relação entre os anos de experiência e a pretensão salarial dos candidatos? Em Python puro, calcular correlação é trabalhoso. Em NumPy é uma linha.

**Conceito de NumPy:** `np.corrcoef(a, b)` retorna uma matriz 2x2 com os coeficientes de correlação. O valor que interessa está na posição `[0, 1]`:

```
np.corrcoef(arr1, arr2)[0, 1]
```

O coeficiente vai de **-1** (correlação negativa perfeita) a **+1** (positiva perfeita), passando por **0** (sem relação linear).

Implemente `correlacao_exp_salario(banco)` que retorna o coeficiente de correlação entre `anos_experiencia` e `pretensao_salarial`, arredondado para 2 casas decimais.

**Exemplo:**

```
correlacao_exp_salario(banco_exemplo)  ->  0.99
```

(o banco de exemplo tem correlação muito alta porque os dados são quase lineares)

In [ ]:
def correlacao_exp_salario(banco):
    arr_exp = extrair_experiencia(banco)
    arr_sal = extrair_salarios(banco)
    return np.corrcoef(arr_exp, arr_sal)

In [ ]:
import numpy as np
# Testes
corr_matrix = correlacao_exp_salario(banco_exemplo)
corr = round(corr_matrix[0, 1], 2) # Extract the relevant coefficient and round
print(0.98 <= corr <= 1.0)

# No banco grande (dados aleatórios), a correlação deve ser próxima de zero
corr_aleatorio_matrix = correlacao_exp_salario(banco_grande)
corr_aleatorio = round(corr_aleatorio_matrix[0, 1], 2) # Extract and round
print(abs(corr_aleatorio) < 0.1)

True
True


In [ ]:
corr_matrix

array([[1.        , 0.99341602],
       [0.99341602, 1.        ]])

**Sugestão de commit:**


feat: adiciona analise de correlacao experiencia x salario

Calcula o coeficiente de Pearson entre anos de experiencia e
pretensao salarial. Permite responder: 'quem tem mais anos de
experiencia pretende mais salario?' em uma unica linha. Em
Python puro essa funcao exigiria mais de 20 linhas.
